# The Research Agent

**Docker image**: `ml4t`

The first three notebooks built the pieces separately: a model client, a search tool with a
cutoff and a source policy, and a state record with gates over it. This notebook assembles
them into one object that takes a question and returns a probability with everything behind
it attached, which is the unit every later notebook in the chapter operates on.

**Learning Objectives**:
- Assemble a model client, a search tool and a turn budget into one agent that answers a
  question end to end
- Parse and validate a model reply that arrives fenced, prefixed, or in the wrong shape, and
  return it for correction instead of failing
- Derive confidence, sentiment, evidence class and key findings from a forecast in code, so
  that every agent's metadata is produced the same way
- Tell an agent that committed to a probability apart from one that ran out of turns, and
  keep the second out of any average
- Read a saved run record and reproduce a forecast made months earlier with no API calls

**Book Reference**: Chapter 24, Section 24.6 (Designing the research agent at the heart of
the pipeline)

**Prerequisites**: [`01_react_reasoning`](01_react_reasoning.ipynb) (providers),
[`02_tool_contracts`](02_tool_contracts.ipynb) (tools),
[`03_state_and_memory`](03_state_and_memory.ipynb) (state and gates).

<!-- ML4T_ZH_BEGIN c0000-p000 -->
# 研究代理

**Docker 映像檔**：`ml4t`

前三份 Notebook 分別建立了各項元件：模型用戶端、具有截止日期及來源政策的搜尋工具，以及設有門檻的狀態紀錄。本 Notebook 將它們組合成一個物件，接收問題並回傳機率，同時附上支撐結果的所有資訊。這個物件是本章後續每份 Notebook 的操作單位。

**學習目標**：
- 將模型用戶端、搜尋工具與輪次預算整合成一個能端到端回答問題的代理
- 解析並驗證被程式碼框包住、帶有前綴，或格式不符的模型回覆；將回覆送回模型修正，而非直接失敗
- 以程式碼從預測推導信心程度、情緒傾向、證據類別及關鍵發現，使每個代理的中繼資料都以相同方式產生
- 區分已給出機率的代理與用盡輪次的代理，並避免將後者納入平均
- 閱讀已儲存的執行紀錄，無須呼叫 API 即可重現數月前做出的預測

**書籍參考**：第 24 章，第 24.6 節（設計預測流程核心的研究代理）

**先備知識**：[`01_react_reasoning`](01_react_reasoning.ipynb)（供應者）、[`02_tool_contracts`](02_tool_contracts.ipynb)（工具）、[`03_state_and_memory`](03_state_and_memory.ipynb)（狀態與門檻）。
<!-- ML4T_ZH_END c0000-p000 -->


In [2]:
"""The Research Agent - ReAct loop with rich output extraction."""

import sys

from utils.paths import get_chapter_dir

sys.path.insert(0, str(get_chapter_dir(24)))

import json
import re
from datetime import date

import polars as pl
from agent_fixtures import get_chapter_contested_question
from agent_observability import (
    TRACES_DIR,
    RunTrace,
    merge_calls,
    show_agent_timeline,
    trace_llm,
)
from agent_pipeline import validate_probabilities
from agent_providers import ChatMessage, LLMClient, TokenUsage, create_llm_client
from agent_schemas import (
    AgentForecastArtifact,
    AgentTrace,
    EvidenceQuality,
    ForecastQuestion,
    SearchResult,
    Sentiment,
)
from agent_tools import (
    MockSearchClient,
    SearchClient,
    ToolExecutor,
    create_search_client,
    eligible_results,
    format_search_results,
)

In [3]:
# RUN_LIVE=False (the default) replays the pinned 2026-06-09 trace named below:
# the notebook reloads that saved run and makes no API calls, so the agent
# outputs are stable and match the chapter. Set RUN_LIVE=True (with API keys) to
# forecast a current question live; that path produces different numbers.
RUN_LIVE = False
PINNED_TRACE = "04_research_agent_20260609T141730Z_b694ab4d0453.json"

LLM_PROVIDER = ""  # Blank reads .env; "mock" selects a synthetic fixture
MAX_STEPS = 5
MAX_SEARCH_RESULTS = 5

## Prompt Templates

The system prompt fixes the agent's role and its two hard constraints: emit valid JSON, and
do not go looking for the market's own price. The second matters for evaluation. An agent
that reads the prediction market it is being scored against is copying, and the copy will
score well, so the constraint is stated to the model and then verified against the saved
prompts further down rather than trusted.

<!-- ML4T_ZH_BEGIN c0003-p000 -->
## 提示詞範本

系統提示詞固定代理的角色，以及兩項硬性限制：輸出有效 JSON，且不要查找預測市場本身的價格。第二項限制對評估至關重要。若代理讀取它所受評分的預測市場，就是在抄答案，而抄來的結果會得到高分。因此，除了在提示詞中告知模型這項限制，下方還會依已儲存的提示詞加以驗證，而不只是信任模型遵守。
<!-- ML4T_ZH_END c0003-p000 -->


In [4]:
AGENT_SYSTEM_PROMPT = """\
You are a forecasting agent in a multi-agent forecasting system.

Your job:
1) Gather evidence by issuing web/news search queries when needed.
2) Then produce a binary probability forecast for the question.

You must follow the action schema exactly and output valid JSON only.
You must not browse prediction market prices unless they are explicitly provided.
Cite a retained source URL or title in your rationale. If evidence is insufficient,
return {"action":"abstain","rationale":"what is missing"}."""

### Step prompt

Provides the question, optional market context, and the action schema.
The agent must output exactly one JSON action per step.

<!-- ML4T_ZH_BEGIN c0005-p000 -->
### 步驟提示詞

提供問題、選用的市場脈絡及行動綱要。代理每一步都必須只輸出一個 JSON 行動。
<!-- ML4T_ZH_END c0005-p000 -->


In [5]:
def build_step_prompt(
    question: ForecastQuestion,
    market_price: float | None = None,
    max_steps: int | None = None,
) -> str:
    """Format the step prompt with question context.

    When ``max_steps`` is given, the agent is told its search budget and is
    nudged to commit once it has enough evidence - searching is not free, and an
    agent that knows its budget stops reformulating the same query and forecasts.
    """
    prompt = f"QUESTION:\n{question.question}\n\n"
    if question.description:
        prompt += f"MARKET CONTEXT:\n{question.description}\n\n"
    if market_price is not None:
        prompt += f"MARKET IMPLIED PROBABILITY (p_yes):\n{market_price}\n\n"
    if max_steps is not None:
        prompt += (
            f"SEARCH BUDGET: you have at most {max_steps} steps. Search only when a "
            "specific, named fact is missing - do not reformulate a query you already "
            "ran. As soon as you have a base rate and the current signal, forecast.\n\n"
        )
    prompt += (
        "NEXT ACTION SCHEMA (output JSON only):\n"
        'If you need more info:\n{"action":"search","query":"..."}\n'
        "If you are ready to forecast:\n"
        '{"action":"forecast","p_yes":0.XX,"rationale":"short explanation '
        'grounded in evidence and base rates"}\n\n'
        "Pick exactly one action."
    )
    return prompt

## JSON Parsing

LLMs sometimes wrap JSON in markdown code blocks or add trailing text.
Robust parsing tries `json.loads` first, then falls back to extracting
the first `{...}` block.

<!-- ML4T_ZH_BEGIN c0007-p000 -->
## JSON 解析

LLM 有時會將 JSON 包在 Markdown 程式碼區塊中，或在後面加入文字。穩健的解析方式會先嘗試 `json.loads`，若失敗則擷取第一個 `{...}` 區塊。
<!-- ML4T_ZH_END c0007-p000 -->


In [6]:
def parse_json(raw: str) -> dict:
    """Parse JSON from LLM output with fallback bracket extraction."""
    raw = raw.strip()
    try:
        parsed = json.loads(raw)
        return (
            parsed
            if isinstance(parsed, dict)
            else {"action": "invalid", "error": "Expected JSON object"}
        )
    except json.JSONDecodeError:
        pass

    # Strip markdown code fences
    raw = re.sub(r"^```(?:json)?\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)
    try:
        parsed = json.loads(raw)
        return (
            parsed
            if isinstance(parsed, dict)
            else {"action": "invalid", "error": "Expected JSON object"}
        )
    except json.JSONDecodeError:
        pass

    # Extract first {...} block
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if match:
        try:
            parsed = json.loads(match.group())
            return (
                parsed
                if isinstance(parsed, dict)
                else {"action": "invalid", "error": "Expected JSON object"}
            )
        except json.JSONDecodeError:
            pass

    return {"action": "invalid", "error": "JSON parse failure"}

## Rich Output Extraction

After the agent produces a forecast, we extract additional metadata from the
raw LLM output. These fields enrich the `AgentForecastArtifact` with
confidence, sentiment, key findings, uncertainties, and evidence quality.

<!-- ML4T_ZH_BEGIN c0009-p000 -->
## 擷取豐富的輸出資訊

代理產生預測後，我們會從原始 LLM 輸出中擷取其他中繼資料。這些欄位為 `AgentForecastArtifact` 增添信心程度、情緒傾向、關鍵發現、不確定性及證據品質資訊。
<!-- ML4T_ZH_END c0009-p000 -->


### Confidence extraction

If the forecast JSON includes a `confidence` field, we use it directly (clamped
to $[0, 1]$). Otherwise, we infer confidence from probability extremity:
$\text{confidence} = 2 \cdot |p_{\text{yes}} - 0.5|$.

<!-- ML4T_ZH_BEGIN c0010-p000 -->
### 信心程度擷取

若預測 JSON 包含 `confidence` 欄位，就直接使用該值（並將其限制在 $[0, 1]$ 範圍內）。否則依機率偏離中間值的程度推估信心：$\text{confidence} = 2 \cdot |p_{\text{yes}} - 0.5|$。
<!-- ML4T_ZH_END c0010-p000 -->


In [7]:
def extract_confidence(action: dict) -> float:
    """Extract confidence from action dict or infer from probability."""
    if "confidence" in action:
        try:
            return max(0.0, min(1.0, float(action["confidence"])))
        except (TypeError, ValueError):
            pass
    p = float(action.get("p_yes", 0.5))
    return round(abs(p - 0.5) * 2, 3)

## Parsing and Validating the Model's Reply

A model told to emit JSON emits JSON most of the time. The rest of the time it wraps the
object in a markdown fence, adds a sentence of explanation before it, or returns something
well-formed that is not an action. All three are ordinary, none is an error condition, and
an agent that crashes on them is an agent that cannot be run unattended.

`parse_json` accepts exactly one JSON object, fenced or bare, and turns anything else into an
explicit parse-failure action. `validate_action` then checks the object against the two
documented schemas: a search needs a non-empty string query, a forecast needs a finite
numeric `p_yes` and a string rationale, and probabilities are clamped to $[0, 1]$ before
anything acts on them. A reply failing either check costs a turn and is returned to the model
for correction.

Both come from `agent_research.py` rather than being written here, because every agent in the
chapter has to reject the same shapes the same way: a parser that differs between notebooks
is a source of disagreement that looks like a difference of judgement.

<!-- ML4T_ZH_BEGIN c0012-p000 -->
## 解析與驗證模型回覆

模型收到輸出 JSON 的指示後，大多數時候都會照做。其他時候，它會將物件包在 Markdown 程式碼框中、在前面加上一句說明，或回傳格式正確但並非行動的內容。這三種都是常見情況，並非異常；若代理因此當機，就無法無人值守地執行。

`parse_json` 只接受一個 JSON 物件，可以有程式碼框，也可以沒有；其他輸入都會轉成明確的解析失敗行動。接著 `validate_action` 依照兩種已記載的綱要檢查物件：搜尋需要非空字串查詢；預測需要有限的數值 `p_yes` 與字串理由；在採取任何行動前，機率會被限制在 $[0, 1]$ 範圍內。若回覆未通過任一檢查，就會耗掉一輪，並送回模型修正。

兩項處理都來自 `agent_research.py`，而非在此重新撰寫，因為本章所有代理都必須以相同方式拒絕相同的錯誤格式。若各份 Notebook 的解析器不同，就可能產生看似判斷差異、實為解析差異的分歧。
<!-- ML4T_ZH_END c0012-p000 -->


## Derived Fields

The model returns a probability and a rationale. Everything else on the artifact is computed
from those two by the functions below. Keeping the derivation in code rather than asking the
model for it means every agent's metadata is produced the same way, which is what makes
agents comparable; it also means the fields are exactly as good as their definitions, and
several of these definitions are crude on purpose. They are labelled as heuristics wherever
they are printed.

<!-- ML4T_ZH_BEGIN c0013-p000 -->
## 衍生欄位

模型回傳機率與理由。產物上的其他資訊，皆由下方函式根據這兩者計算。將推導過程寫在程式碼中，而非要求模型自行提供，代表每個代理的中繼資料都以相同方式產生，代理之間才可比較；同時也代表這些欄位的品質完全取決於其定義，其中幾項定義刻意保持粗略。凡是印出這些欄位時，都會標示它們是啟發式結果。
<!-- ML4T_ZH_END c0013-p000 -->


### Confidence

When the model volunteers a `confidence` field it is used, clamped to $[0, 1]$. Otherwise
confidence is taken as distance from even odds,
$\text{confidence} = 2\,\lvert p_{\text{yes}} - \tfrac{1}{2} \rvert$, which is zero at
even odds and one at either certainty.

That is a statement about the probability's position, not about the evidence behind it. An
agent that read forty documents and concluded the question is genuinely balanced scores zero,
and so does an agent that read nothing and guessed at even odds. The number is cheap and it
means the same thing for every agent, which is what it is for. What would answer the question
instead is calibration: whether probabilities stated at seventy percent come true about
seventy percent of the time.
[`09_evaluation_and_governance`](09_evaluation_and_governance.ipynb) builds that arithmetic
and runs it on synthetic inputs, because measuring it for real needs forecasts recorded
before their questions resolved, which no capture in this chapter has.

<!-- ML4T_ZH_BEGIN c0014-p000 -->
### 信心程度

若模型主動提供 `confidence` 欄位，就使用該值，並限制在 $[0, 1]$ 範圍內。否則，信心程度取自機率偏離五五波的距離，即 $\text{confidence} = 2\,\lvert p_{\text{yes}} - \tfrac{1}{2} \rvert$；五五波時為零，任一方向完全確定時為一。

這描述的是機率所處的位置，而非背後的證據。讀過四十份文件後認為問題確實勢均力敵的代理，得分為零；什麼都沒讀、猜了五五波的代理也一樣。這個數值計算成本低，對每個代理的意義也相同，這就是它的用途。真正能回答信心問題的是校準度：宣稱機率為七成的事件，是否約有七成真的發生。[`09_evaluation_and_governance`](09_evaluation_and_governance.ipynb) 會建立相關計算，並在合成輸入上執行；但要實際衡量校準度，需要在問題結算之前記錄預測，本章的擷取紀錄都不具備這項條件。
<!-- ML4T_ZH_END c0014-p000 -->


### Sentiment extraction

Maps $p_{\text{yes}}$ mechanically to a five-level sentiment scale.

<!-- ML4T_ZH_BEGIN c0015-p000 -->
### 情緒傾向擷取

以固定規則將 $p_{\text{yes}}$ 對應至五級情緒傾向量表。
<!-- ML4T_ZH_END c0015-p000 -->


In [8]:
def extract_sentiment(p_yes: float) -> Sentiment:
    """Infer sentiment from probability."""
    if p_yes > 0.8:
        return Sentiment.STRONGLY_BULLISH
    if p_yes > 0.6:
        return Sentiment.BULLISH
    if p_yes > 0.4:
        return Sentiment.NEUTRAL
    if p_yes > 0.2:
        return Sentiment.BEARISH
    return Sentiment.STRONGLY_BEARISH

### Key findings

A model asked for a short rationale usually enumerates its reasons, and it picks the format
on its own: sometimes one item per line behind a dash or a number, sometimes run together
inside a sentence as `(1) ... (2) ...`. The function takes both, preferring line-leading
items where they exist and falling back to the inline markers, and it requires at least two
markers so that a lone parenthesised digit in ordinary prose is not read as a list.

A rationale that enumerates nothing yields nothing, and that is the honest outcome rather
than a failure: the field records how the model chose to present its reasons, not how many
reasons it had.

The last inline item is the one that needs care, because the text after the final marker
runs on into whatever the model wrote next, so the item has to end at its own sentence
boundary. Finding that boundary means telling a full stop from an abbreviation:
*U.S. inflation remains elevated* is one sentence, and a rule that cut at every stop would
return `U.S`. The item ends only where the letter before the stop is not a capital and the
next word begins with one.

<!-- ML4T_ZH_BEGIN c0017-p000 -->
### 關鍵發現

要求模型提供簡短理由時，它通常會列舉原因，並自行選擇格式：有時每項前面加上破折號或編號、各占一行；有時像 `(1) ... (2) ...` 一樣連寫在句子中。函式同時處理兩種形式：優先使用行首項目，若沒有再使用行內標記；而且要求至少兩個標記，以免將一般敘述中偶然出現的單個括號數字誤認為清單。

若理由沒有列舉內容，結果就是空的；這是如實呈現，而非失敗：該欄記錄的是模型如何表達其理由，不是理由有多少項。

最後一個行內項目需要特別小心，因為最終標記後的文字會接著模型的後續敘述，所以該項目必須在自己的句子邊界結束。找出邊界時，需要區分句點與縮寫：*U.S. inflation remains elevated* 是一個句子，若規則在每個句點都截斷，就會回傳 `U.S`。只有當句點前的字母不是大寫，且下一個單字以大寫開頭時，該項目才結束。
<!-- ML4T_ZH_END c0017-p000 -->


In [9]:
def extract_key_findings(rationale: str) -> list[str]:
    """Extract bullet points and numbered items from rationale."""
    findings = []
    for line in rationale.split("\n"):
        line = line.strip()
        if re.match(r"^[-•*]\s+", line):
            findings.append(re.sub(r"^[-•*]\s+", "", line).strip())
        elif re.match(r"^\d+[.)]\s+", line):
            findings.append(re.sub(r"^\d+[.)]\s+", "", line).strip())
    return findings[:10]

### Uncertainty extraction

Identifies sentences containing uncertainty language in the rationale.

<!-- ML4T_ZH_BEGIN c0019-p000 -->
### 不確定性擷取

辨識理由中含有不確定性用語的句子。
<!-- ML4T_ZH_END c0019-p000 -->


In [10]:
def extract_uncertainties(rationale: str) -> list[str]:
    """Identify sentences mentioning uncertainty."""
    uncertainty_words = {
        "uncertain",
        "unclear",
        "unknown",
        "risk",
        "caveat",
        "however",
        "but",
        "although",
    }
    sentences = re.split(r"[.!?]+", rationale)
    return [
        s.strip() for s in sentences if s.strip() and any(w in s.lower() for w in uncertainty_words)
    ][:5]

### Evidence quality assessment

This volume heuristic uses only query and result counts. It does not validate
source credibility, independence, or point-in-time availability.

<!-- ML4T_ZH_BEGIN c0021-p000 -->
### 證據品質評估

這項證據量啟發式規則只使用查詢與結果數量，不會驗證來源可信度、獨立性，或在特定時間點是否可取得。
<!-- ML4T_ZH_END c0021-p000 -->


In [11]:
def assess_evidence_quality(sources_consulted: int, queries_made: int) -> EvidenceQuality:
    """Assess evidence quality from search metrics."""
    if sources_consulted >= 10 and queries_made >= 3:
        return EvidenceQuality.HIGH
    if sources_consulted >= 5 or queries_made >= 2:
        return EvidenceQuality.MEDIUM
    return EvidenceQuality.LOW

## ReAct step handlers

The ReAct loop processes one of three action types per step: a search,
a forecast, or an unrecognised action. Extracting each branch as a small
free function keeps the agent class focused on iteration and assembly.

<!-- ML4T_ZH_BEGIN c0023-p000 -->
## ReAct 步驟處理器

ReAct 迴圈每一步會處理三種行動之一：搜尋、預測或無法辨識的行動。將各分支提取為小型獨立函式，可讓代理類別專注於迭代與組裝。
<!-- ML4T_ZH_END c0023-p000 -->


In [12]:
# Check the returned sources, including supplied context.

## The ResearchAgent Class

The agent owns the LLM, the search executor, and the iteration budget.
Its `run()` method walks the ReAct loop, dispatches each step to the
handlers above, and returns an `AgentForecastArtifact` with the
accumulated metadata. The class is reused unchanged in NB06 (multi-agent)
and NB08 (full pipeline).

<!-- ML4T_ZH_BEGIN c0025-p000 -->
## ResearchAgent 類別

代理持有 LLM、搜尋執行器及迭代預算。其 `run()` 方法逐步執行 ReAct 迴圈，將每一步交由上述處理器處理，並回傳附有累積中繼資料的 `AgentForecastArtifact`。NB06（多代理）與 NB08（完整流程）會原樣重用此類別。
<!-- ML4T_ZH_END c0025-p000 -->


In [13]:
class ResearchAgent:
    """ReAct research agent producing evidence-linked probability forecasts.

    Implements the same loop as the AIA Forecaster's research agent:
    search for evidence, then forecast with rich metadata.
    """

    def __init__(
        self,
        llm: LLMClient,
        search: SearchClient | None = None,
        agent_id: str = "agent_0",
        max_steps: int = 6,
        max_search_results: int = 5,
    ) -> None:
        self.llm = llm
        self.executor = ToolExecutor(search=search)
        self.agent_id = agent_id
        self.max_steps = max_steps
        self.max_search_results = max_search_results

    def run(
        self,
        question: ForecastQuestion,
        market_price: float | None = None,
        evidence: list[SearchResult] | None = None,
    ) -> AgentForecastArtifact:
        """Research within the budget; retain unsupported attempts as abstentions."""
        if self.max_steps < 1 or self.max_search_results < 1:
            raise ValueError("Research budgets must be positive")
        cutoff = date.fromisoformat(question.cutoff_date) if question.cutoff_date else None
        messages = [
            ChatMessage(role="system", content=AGENT_SYSTEM_PROMPT),
            ChatMessage(
                role="user", content=build_step_prompt(question, market_price, self.max_steps)
            ),
        ]
        traces: list[AgentTrace] = []
        sources = eligible_results(evidence or [], cutoff)
        if sources:
            traces.append(AgentTrace(step=0, action="supplied_evidence", results=sources))
            messages.append(ChatMessage(role="user", content=format_search_results(sources)))
        total_tokens = TokenUsage()
        queries_made = 0
        p_yes = None
        rationale = ""
        raw_action = {}
        reason = "Budget exhausted without a supported forecast"
        status = "abstained"

        for step in range(1, self.max_steps + 2):
            synthesis_only = step > self.max_steps
            if synthesis_only:
                if not sources:
                    break
                messages.append(
                    ChatMessage(
                        role="user",
                        content=(
                            "Search budget exhausted. Use only the retained evidence to forecast, "
                            "citing its URL or title. If it is insufficient, abstain. No further search."
                        ),
                    )
                )
            try:
                response, usage = self.llm.complete_with_usage(messages, json_mode=True)
            except Exception as exc:
                reason = f"Model call failed ({type(exc).__name__})"
                status = "failed"
                traces.append(AgentTrace(step=step, action="provider_error", llm_raw=reason))
                break
            total_tokens = total_tokens + usage
            messages.append(ChatMessage(role="assistant", content=response))
            action = parse_json(response)
            action_type = action.get("action", "invalid")
            if action_type == "search" and not synthesis_only:
                query = action.get("query", "")
                results = self.executor.execute_search(query, self.max_search_results, cutoff)
                log = self.executor.execution_log[-1]
                queries_made += int(log.status in {"success", "error"})
                sources.extend(results)
                traces.append(
                    AgentTrace(
                        step=step, action="search", query=query, results=results, llm_raw=response
                    )
                )
                messages.append(
                    ChatMessage(
                        role="tool",
                        content=(
                            format_search_results(results)
                            if results
                            else f"No eligible evidence. Search status: {log.status}. {log.result_preview}"
                        ),
                    )
                )
                continue
            traces.append(AgentTrace(step=step, action=str(action_type), llm_raw=response))
            if action_type == "abstain":
                reason = str(action.get("rationale") or "Model reported insufficient evidence")
                break
            if action_type == "forecast":
                try:
                    if isinstance(action["p_yes"], bool):
                        raise ValueError("Probability must be a number, not a boolean")
                    candidate = float(action["p_yes"])
                    validate_probabilities([candidate])
                except (KeyError, TypeError, ValueError):
                    reason = "Invalid probability; forecast rejected"
                else:
                    explanation = action.get("rationale")
                    if not sources:
                        reason = "No eligible evidence; forecast rejected"
                    elif not isinstance(explanation, str) or not explanation.strip():
                        reason = "Missing rationale; forecast rejected"
                    elif not any(
                        (r.url and r.url in explanation) or (r.title and r.title in explanation)
                        for r in sources
                    ):
                        reason = "Rationale must identify a retained source URL or title"
                    else:
                        p_yes, rationale, raw_action = candidate, explanation, action
                        status = "accepted"
                        break
            else:
                reason = "Invalid action or search requested after the budget was exhausted"
            messages.append(ChatMessage(role="tool", content=reason))

        return AgentForecastArtifact(
            agent_id=self.agent_id,
            p_yes=p_yes,
            rationale=rationale,
            traces=traces,
            status=status,
            failure_reason=None if status == "accepted" else reason,
            messages=[{"role": m.role, "content": m.content} for m in messages],
            execution_mode=(
                "synthetic"
                if isinstance(self.executor.search, MockSearchClient)
                or any(
                    label in getattr(self.llm, "model_name", "").lower()
                    for label in ("mock", "synthetic")
                )
                else "live"
            ),
            confidence=extract_confidence(raw_action) if p_yes is not None else 0.0,
            sentiment=extract_sentiment(p_yes) if p_yes is not None else Sentiment.NEUTRAL,
            key_findings=extract_key_findings(rationale),
            evidence_quality=assess_evidence_quality(len(sources), queries_made),
            uncertainties=extract_uncertainties(rationale),
            token_usage=total_tokens,
            search_queries_made=queries_made,
            sources_consulted=len(sources),
        )

## Running the Research Agent

The question is `CHAPTER_CONTESTED_QUESTION` from `agent_fixtures.py`: *"Will the Federal
Reserve hike rates in 2026?"*. It is contested in the specific sense that matters here, which
is that two runs of the same agent reach opposite conclusions on it, so it is the question
the chapter uses wherever disagreement is the subject.
[`07_adversarial_debate`](07_adversarial_debate.ipynb) makes agents argue about it and
[`08_forecasting_pipeline`](08_forecasting_pipeline.ipynb) runs the full pipeline on it.
[`06_multi_agent_research`](06_multi_agent_research.ipynb) uses the companion
`CHAPTER_CLEAR_QUESTION` instead, where the agents agree, so the two can be compared.

On the default path the notebook replays the pinned capture rather than calling anything;
the cell below reports the provider, the search tool and the date of the capture it read,
each taken from the saved record. Setting `RUN_LIVE = True` with `ANTHROPIC_API_KEY` and
`TAVILY_API_KEY` forecasts a current question instead, and will not reproduce the values
below.

<!-- ML4T_ZH_BEGIN c0027-p000 -->
## 執行研究代理

問題是 `CHAPTER_CONTESTED_QUESTION`，來自 `agent_fixtures.py`：*「聯準會會在 2026 年升息嗎？」* 此問題之所以有爭議，具體而言，是因為同一代理的兩次執行得出相反結論；因此本章凡是討論分歧，都會使用這個問題。[`07_adversarial_debate`](07_adversarial_debate.ipynb) 讓代理就此辯論，[`08_forecasting_pipeline`](08_forecasting_pipeline.ipynb) 則以它執行完整流程。[`06_multi_agent_research`](06_multi_agent_research.ipynb) 改用另一個 `CHAPTER_CLEAR_QUESTION`，代理在該問題上的看法一致，讓兩者可以比較。

預設路徑會重播固定的擷取紀錄，不呼叫外部服務；下方儲存格報告供應者、搜尋工具及所讀擷取紀錄的日期，三者都來自已儲存的紀錄。若設定 `RUN_LIVE = True`，並提供 `ANTHROPIC_API_KEY` 與 `TAVILY_API_KEY`，則會改為預測當前問題，且不會重現下方數值。
<!-- ML4T_ZH_END c0027-p000 -->


In [14]:
if RUN_LIVE:
    llm = create_llm_client(LLM_PROVIDER)
    search = create_search_client()

    # Wrap the client in a TracingLLMClient so every prompt and raw response is
    # captured for the run trace we persist at the end of the notebook.
    tracer_0 = trace_llm(llm, label="agent_0")
    agent = ResearchAgent(
        llm=tracer_0,
        search=search,
        agent_id="agent_0",
        max_steps=MAX_STEPS,
        max_search_results=MAX_SEARCH_RESULTS,
    )

    question = get_chapter_contested_question()
    provider_name = llm.model_name
    search_name = type(search).__name__
    artifact = agent.run(question)
else:
    # Replay: reload the pinned trace and rehydrate its two saved agents. The
    # first is the single-agent forecast inspected here; the second is the
    # preview pair shown later. Display cells consume these unchanged.
    pinned_run = RunTrace.load(TRACES_DIR / PINNED_TRACE)
    question = pinned_run.question_obj()
    provider_name = pinned_run.provider
    search_name = "replay (pinned trace)"
    replayed_artifacts = pinned_run.agent_artifacts()
    artifact = replayed_artifacts[0]

print(f"Mode: {'LIVE' if RUN_LIVE else 'REPLAY (pinned 2026-06-09 trace)'}")
print(f"Provider: {provider_name}")
print(f"Search: {search_name}")
print(f"Question: {question.question}\n")

Mode: REPLAY (pinned 2026-06-09 trace)
Provider: claude-sonnet-4-20250514
Search: replay (pinned trace)
Question: Will the Federal Reserve hike rates in 2026?



## Inspecting the Forecast Artifact

One object holds everything the run produced. The first three fields are what the model
committed to; the rest were computed from them by the extraction functions above, and are
labelled as heuristics wherever they are printed so that a derived class is never read as a
measurement.

<!-- ML4T_ZH_BEGIN c0029-p000 -->
## 檢查預測產物

單一物件保存這次執行產生的全部內容。前三個欄位是模型明確給出的判斷；其他欄位則由上述擷取函式根據這些資訊計算，且印出時會標示為啟發式結果，以免將衍生類別誤認為測量值。
<!-- ML4T_ZH_END c0029-p000 -->


In [15]:
print("=== Forecast ===")
print(f"Agent:       {artifact.agent_id}")
print(f"Status:      {artifact.status}")
print(f"p(YES):      {artifact.p_yes}")
if artifact.failure_reason:
    print(f"Reason:      {artifact.failure_reason}")
print(f"Confidence:  {artifact.confidence:.2f}")
print(f"Sentiment:   {artifact.sentiment.value}")
print(f"Evidence:    {artifact.evidence_quality.value}")
print(f"Queries:     {artifact.search_queries_made}")
print(f"Sources:     {artifact.sources_consulted}")
print(f"Tokens:      {artifact.token_usage.total_tokens:,}")

=== Forecast ===
Agent:       agent_0
Status:      legacy_capture
p(YES):      0.65
Confidence:  0.30
Sentiment:   bullish
Evidence:    high
Queries:     4
Sources:     20
Tokens:      23,335


In [16]:
print(f"\nRationale:\n{artifact.rationale[:400]}")


Rationale:
Based on current evidence, there's a moderate-to-high probability of Fed rate hikes in 2026. Key factors: (1) Markets now price >50% chance of rate hikes by end-2026 due to inflation concerns from Iran war and rising oil prices ($110+ crude); (2) Fed currently holds rates at 4.25-4.5% after cutting 1.75% in 2024-2025; (3) Inflation pressures mounting from energy costs and tariff impacts; (4) Histo


In [17]:
if artifact.key_findings:
    print("\nKey Findings:")
    for f in artifact.key_findings:
        print(f"  • {f}")

if artifact.uncertainties:
    print("\nUncertainties:")
    for u in artifact.uncertainties:
        print(f"  • {u}")


Uncertainties:
  • However, uncertainty remains around economic growth, labor market conditions, and geopolitical developments that could shift policy direction


## Execution Trace

Every search query and its results are captured in the trace. `show_agent_timeline`
from `agent_observability` renders the whole run in order: each query, the
documents it returned (title, date, URL, and a snippet), and the forecast with
the untruncated rationale. This is the per-agent observability view reused
across the multi-agent notebooks that follow.

<!-- ML4T_ZH_BEGIN c0033-p000 -->
## 執行軌跡

每項搜尋查詢及其結果都會保存在軌跡中。`show_agent_timeline` 來自 `agent_observability`，會依序呈現整次執行：每項查詢、回傳的文件（標題、日期、URL 及摘要片段），以及附有完整理由、未經截斷的預測。後續多代理 Notebook 會重用這份個別代理的可觀察性檢視。
<!-- ML4T_ZH_END c0033-p000 -->


In [18]:
print(show_agent_timeline(artifact))

  AGENT_0  ·  p_yes=0.65  status=legacy_capture  confidence=0.30  sentiment=bullish  evidence=high
  searches=4  sources=20  tokens=23,335
  Step 1 · SEARCH  "Federal Reserve interest rates 2024 2025 current policy outlook"
      1. [?] Federal Reserve keeps interest rates steady as inflation uncertainty ...
         https://www.usbank.com/investing/financial-perspectives/market-news/federal-reserve-interest-rate.html
         The Fed shifted as inflation slowed, cutting its target interest rate
         by 1.75% through 2024 and 2025. Oil prices have increased more than
         60%
      2. [?] United States Interest Rate Outlook & Forecast - FocusEconomics
         https://www.focus-economics.com/country-indicator/united-states/interest-rate
         The federal funds target rate ended 2024 at 4.50%, compared to the
         end-2023 value of 5.50% and the figure a decade earlier of 0.25%. It
         averaged 2.02% over the
      3. [?] Federal Reserve Cuts Interest Rates in Late 2

## Tool Execution Audit

The timeline above is assembled from the agent's own traces, which is the model's account of
the run. `ToolExecutor` keeps a second, independent record of what it was actually asked to
do. On a live run that log carries a per-call `duration_ms`; the pinned trace does not
persist wall-clock timing, because a duration measured on one machine reproduces on no other,
so the replay path rebuilds the query, status and result count from the saved traces instead.

A table rather than aligned text: the same three fields, in a form that sorts, filters and
joins without anyone parsing a string.

<!-- ML4T_ZH_BEGIN c0035-p000 -->
## 工具執行稽核

上方時間軸是根據代理自己的軌跡組成，也就是模型對這次執行的描述。`ToolExecutor` 另外保存一份獨立紀錄，記下它實際收到哪些工作要求。即時執行時，該紀錄包含每次呼叫的 `duration_ms`；固定軌跡不保存實際耗時，因為在某台機器上測得的時間無法在其他機器上重現，所以重播路徑改從已儲存的軌跡重建查詢、狀態與結果數量。

此處使用表格，而非對齊的文字：同樣三個欄位，以可排序、篩選與合併、無須解析字串的形式呈現。
<!-- ML4T_ZH_END c0035-p000 -->


In [19]:
if RUN_LIVE:
    audit_df = (
        pl.DataFrame(
            [
                {
                    "query": entry.args.get("query", "?"),
                    "status": entry.status,
                    "duration_ms": round(entry.duration_ms, 1),
                }
                for entry in agent.executor.execution_log
            ]
        )
        if agent.executor.execution_log
        else pl.DataFrame({"query": [], "status": [], "duration_ms": []})
    )
else:
    # Replay: reconstruct the search audit from the saved agent traces.
    search_steps = [t for t in artifact.traces if t.action == "search"]
    audit_df = (
        pl.DataFrame(
            [
                {
                    "query": t.query or "?",
                    "status": "ok",
                    "results": len(t.results),
                }
                for t in search_steps
            ]
        )
        if search_steps
        else pl.DataFrame({"query": [], "status": [], "results": []})
    )
audit_df

query,status,results
str,str,i64
"""Federal Reserve interest rates…","""ok""",5
"""Federal Reserve 2026 interest …","""ok""",5
"""Federal Reserve current intere…","""ok""",5
"""Federal Reserve rate hike prob…","""ok""",5


Freshness passes and consistency reports that there is no cutoff to enforce, which is a
statement about the question rather than a defect in the run. `CHAPTER_CONTESTED_QUESTION`
carries `cutoff_date=""` because it was open when it was captured: it asks about the rest of
2026, so no document available on the day could have held the answer and no date needs
excluding. The gate having nothing to check is the correct outcome. It has teeth on the
resolved questions in `agent_fixtures.py`, each of which carries a cutoff, and reading past
one is how a forecasting result gets faked.

Coverage is a different matter, and its failure says more about the adapter than about the
agent. Every item above is tagged `search_results`, because a trace records what the agent
searched for and not what kind of evidence it decided each result was. The coverage gate asks
for a `base_rate` item and cannot find one, and it could not have found one however well the
agent had researched. Reading that as "the agent skipped the base rate" would be reading a
limitation of the reconstruction as a finding about the run.

This is the argument [`03_state_and_memory`](03_state_and_memory.ipynb) makes, arriving from
the other direction: evidence type is a judgement made while gathering, and a record written
afterwards from a trace cannot recover it. An agent that is going to be gated on coverage has
to write its state as it goes, classifying each result as it arrives, rather than have the
classification inferred from a log at the end.

<!-- ML4T_ZH_BEGIN c0037-p000 -->
新近性檢查通過；一致性檢查則回報沒有可執行的截止日。這說明的是問題性質，而非執行缺陷。`CHAPTER_CONTESTED_QUESTION` 帶有 `cutoff_date=""`，因為擷取當時問題尚未結算：它詢問 2026 年剩餘時間的情況，所以當天可取得的任何文件都不可能包含答案，也不需要排除日期。門檻沒有內容可查，是正確結果。對於 `agent_fixtures.py` 中已結算的問題，這道門檻才真正發揮作用：每個問題都有截止日，若讀取截止日後的資料，預測結果便可能造假。

覆蓋度則是另一回事，其失敗更能說明轉接器的限制，而非代理的問題。上方每個項目都標記為 `search_results`，因為軌跡記錄代理搜尋了什麼，而不是代理判斷每項結果屬於哪種證據。覆蓋度門檻要求 `base_rate` 項目，卻找不到；無論代理的研究多充分，也不可能從該軌跡中找到。若將此解讀成「代理略過了基準發生率」，就是把重建方法的限制誤讀成執行結果。

這也是 [`03_state_and_memory`](03_state_and_memory.ipynb) 從另一個方向提出的論點：證據類型是在收集時做出的判斷，事後根據軌跡寫出的紀錄無法還原。若要依覆蓋度門檻檢查代理，就必須在執行過程中同步寫入狀態，並在每項結果抵達時分類，而非最後才從日誌推斷分類。
<!-- ML4T_ZH_END c0037-p000 -->


## Agent Summary Format

When several agents run in parallel, their forecasts are summarized
for the supervisor and debate stages. This is the format used downstream.

<!-- ML4T_ZH_BEGIN c0038-p000 -->
## 代理摘要格式

多個代理平行執行時，會將其預測摘要提供給監督者與辯論階段。這是下游使用的格式。
<!-- ML4T_ZH_END c0038-p000 -->


In [20]:
def format_agent_summary(a: AgentForecastArtifact) -> str:
    """Format an agent artifact as a summary for supervisor/debate prompts."""
    lines = [
        f"Agent: {a.agent_id}",
        f"Probability (p_yes): {a.p_yes}",
        f"Status: {a.status}",
        f"Confidence: {a.confidence:.2f}",
        f"Rationale: {a.rationale}",
    ]
    if a.key_findings:
        lines.append("Key findings:")
        for f in a.key_findings[:3]:
            lines.append(f"  - {f}")
    for trace in a.traces:
        for source in trace.results:
            lines.append(f"Source: {source.title} | {source.url}")
            lines.append(f"Excerpt: {(source.snippet or '')[:400]}")
    return "\n".join(lines)

In [21]:
print(format_agent_summary(artifact))

Agent: agent_0
Probability (p_yes): 0.65
Status: legacy_capture
Confidence: 0.30
Rationale: Based on current evidence, there's a moderate-to-high probability of Fed rate hikes in 2026. Key factors: (1) Markets now price >50% chance of rate hikes by end-2026 due to inflation concerns from Iran war and rising oil prices ($110+ crude); (2) Fed currently holds rates at 4.25-4.5% after cutting 1.75% in 2024-2025; (3) Inflation pressures mounting from energy costs and tariff impacts; (4) Historical precedent shows Fed responds to persistent inflation with rate increases. However, uncertainty remains around economic growth, labor market conditions, and geopolitical developments that could shift policy direction.
Source: Federal Reserve keeps interest rates steady as inflation uncertainty ... | https://www.usbank.com/investing/financial-perspectives/market-news/federal-reserve-interest-rate.html
Excerpt: The Fed shifted as inflation slowed, cutting its target interest rate by 1.75% through 202

## Two Agents, One Question

The same class with a different id gives a second sample. Nothing else differs: same
question, same prompts, same search tool, same model. The two probabilities below come apart
anyway, which is the observation the rest of the chapter is built on. Where that variation
comes from - which documents each search returned, and the model's own sampling - is not
identified here, and [`06_multi_agent_research`](06_multi_agent_research.ipynb) does not
identify it either; what that notebook does is run enough copies for the spread to be worth
looking at.

<!-- ML4T_ZH_BEGIN c0041-p000 -->
## 兩個代理，一個問題

同一個類別使用不同 ID，就能得到第二次取樣。其他條件完全相同：相同問題、提示詞、搜尋工具與模型。然而，下方兩個機率仍有差異；本章其餘部分就是建立在這項觀察之上。這種變異究竟來自每次搜尋回傳的文件，還是模型自身的取樣，此處並未區分；[`06_multi_agent_research`](06_multi_agent_research.ipynb) 也沒有區分。後者做的是執行足夠多次，讓機率分布範圍值得檢視。
<!-- ML4T_ZH_END c0041-p000 -->


In [22]:
if RUN_LIVE:
    tracer_1 = trace_llm(llm, label="agent_1")
    agent_b = ResearchAgent(llm=tracer_1, search=search, agent_id="agent_1", max_steps=MAX_STEPS)
    artifact_b = agent_b.run(question)
else:
    # Replay: the pinned trace's second saved agent is this preview run.
    artifact_b = replayed_artifacts[1]

for a in [artifact, artifact_b]:
    print(f"{a.agent_id}: status={a.status}, p_yes={a.p_yes}")
if artifact.p_yes is not None and artifact_b.p_yes is not None:
    print(f"Difference: {abs(artifact.p_yes - artifact_b.p_yes):.2f}")

agent_0: status=legacy_capture, p_yes=0.65
agent_1: status=legacy_capture, p_yes=0.25
Difference: 0.40


**Interpretation**: The captured agents searched and weighted evidence differently.
Their probability spread can reflect retrieval, sampling, prompts and model biases.
These two runs do not isolate those effects or establish independent information.
Notebooks 06 and 07 inspect an ensemble and debate on other questions;
resolved comparable runs are needed to assess whether either improves forecasting.

<!-- ML4T_ZH_BEGIN c0043-p000 -->
**解讀**：擷取紀錄中的代理以不同方式搜尋並權衡證據。它們的機率差異可能反映資訊擷取、取樣、提示詞及模型偏差。這兩次執行無法分離各項影響，也不能證明取得了獨立資訊。Notebook 06 與 07 會在其他問題上檢視集成結果與辯論；要評估其中任一方法是否改善預測，仍需要已結算且可比較的執行紀錄。
<!-- ML4T_ZH_END c0043-p000 -->


## Persisting the Run Trace

The artifact holds the structured forecast; the `TracingLLMClient`s wrapped
around each agent hold the raw conversation. `RunTrace.capture` bundles both -
the question, both agents' artifacts, and every prompt/response - into one JSON
record under `forecast_traces/`, the same auditable format the multi-agent
notebooks (NB06–NB08) write. Reload it with `RunTrace.load` to replay exactly
what the agent saw and said - which is what the default `RUN_LIVE = False` path
does above. A live run writes a fresh trace here; the default replay run reports
the pinned trace it loaded rather than overwriting it.

<!-- ML4T_ZH_BEGIN c0044-p000 -->
## 保存執行軌跡

產物保存結構化的預測；包覆各代理的 `TracingLLMClient` 則保存原始對話。`RunTrace.capture` 將兩者整合，把問題、兩名代理的產物及每組提示詞與回覆，存成 `forecast_traces/` 下的單一 JSON 紀錄；多代理 Notebook（NB06–NB08）也會寫入相同的可稽核格式。使用 `RunTrace.load` 重新載入，即可精確重播代理看到與說出的內容；上方預設的 `RUN_LIVE = False` 路徑正是如此運作。即時執行會在此寫入新的軌跡；預設重播則只報告載入的固定軌跡，不會覆寫它。
<!-- ML4T_ZH_END c0044-p000 -->


In [23]:
if RUN_LIVE:
    run = RunTrace.capture(
        notebook="04_research_agent",
        provider=provider_name,
        question=question,
        params={"max_steps": MAX_STEPS, "max_search_results": MAX_SEARCH_RESULTS},
        agents=[artifact, artifact_b],
        notes="Single research agent plus a two-agent preview on the contested question.",
        llm_calls=merge_calls(tracer_0, tracer_1),
    )
    trace_path = run.save()
    print(
        f"Saved {len(run.llm_calls)} model calls "
        f"({run.total_tokens():,} tokens) → {trace_path.relative_to(trace_path.parents[1])}"
    )
else:
    # Replay: report the pinned trace we loaded rather than writing a new file.
    run = pinned_run
    trace_path = TRACES_DIR / PINNED_TRACE
    print(
        f"Replayed {len(run.llm_calls)} model calls "
        f"({run.total_tokens():,} tokens) from {trace_path.name}"
    )

Replayed 10 model calls (42,266 tokens) from 04_research_agent_20260609T141730Z_b694ab4d0453.json


## Key Takeaways

1. **ResearchAgent** combines the ReAct loop with rich output extraction - every
   forecast comes with confidence, sentiment, evidence quality, and uncertainties
2. **Structured artifacts**: `AgentForecastArtifact` captures the complete run -
   probability, reasoning, traces, and token usage
3. **Robust parsing**: JSON extraction handles markdown code blocks and malformed
   LLM output gracefully
4. **Reusable**: This class is the building block for multi-agent (NB06),
   debate (NB07), and full pipeline (NB08) notebooks
5. **Token tracking**: Every LLM call is metered for cost analysis

**Next**: [`aggregation`](05_aggregation_math.ipynb) - the mathematical foundation for combining
multiple probability estimates (Neyman extremization, weighted aggregation).

**Book**: Section 24.6 discusses agent design patterns, including the trade-off
between agent complexity and forecast calibration.

<!-- ML4T_ZH_BEGIN c0046-p000 -->
## 重點整理

1. **ResearchAgent** 將 ReAct 迴圈與豐富的輸出擷取結合；每份預測都附有信心程度、情緒傾向、證據品質及不確定性
2. **結構化產物**：`AgentForecastArtifact` 保存完整執行，包括機率、推理、軌跡及 token 用量
3. **穩健解析**：JSON 擷取能妥善處理 Markdown 程式碼區塊及格式錯誤的 LLM 輸出
4. **可重用**：此類別是多代理（NB06）、辯論（NB07）及完整流程（NB08）Notebook 的基礎元件
5. **Token 追蹤**：衡量每次 LLM 呼叫的用量，以供成本分析

**下一步**：[`aggregation`](05_aggregation_math.ipynb)：結合多個機率估計值的數學基礎（Neyman 極端化、加權彙整）。

**書籍**：第 24.6 節討論代理設計模式，包括代理複雜度與預測校準度之間的取捨。
<!-- ML4T_ZH_END c0046-p000 -->


### Failure test: a probability without evidence

This scripted model deliberately tries to forecast before acquiring evidence. The actual research loop must retain the attempt and abstain. No live model or invented performance result is involved.

<!-- ML4T_ZH_BEGIN c0047-p000 -->
### 失敗測試：沒有證據的機率

這個預先編排的模型刻意在取得證據之前嘗試預測。實際研究迴圈必須保留這次嘗試的紀錄並棄答。這裡沒有使用即時模型，也沒有虛構績效結果。
<!-- ML4T_ZH_END c0047-p000 -->


In [24]:
class UnsupportedForecastFixture:
    model_name = "synthetic-failure-fixture"

    def complete_with_usage(self, messages, **kwargs):
        return '{"action":"forecast","p_yes":0.9,"rationale":"No sources consulted"}', TokenUsage()


unsupported = ResearchAgent(UnsupportedForecastFixture(), max_steps=1).run(
    ForecastQuestion("Deliberately unsupported teaching forecast")
)
assert unsupported.status == "abstained" and unsupported.p_yes is None
print(unsupported.status, unsupported.failure_reason)

abstained No eligible evidence; forecast rejected


## Fresh verification of the current loop

This October 2, 2026 capture uses genuine Sonnet 4.6 and Tavily calls on an
unresolved October FOMC question. Inspect its retained evidence and model I/O
alongside the original June capture above. The forecast has no outcome score;
acceptance means a valid probability, rationale and eligible cited evidence.

<!-- ML4T_ZH_BEGIN c0049-p000 -->
## 重新驗證目前的迴圈

這份 2026 年 10 月 2 日的擷取紀錄，針對尚未結算的十月 FOMC 問題，使用真正的 Sonnet 4.6 與 Tavily 呼叫。請將其保留的證據及模型輸入輸出，與上方六月的原始擷取紀錄一同檢視。該預測沒有結果分數；驗收代表機率、理由及所引用的合格證據都有效。
<!-- ML4T_ZH_END c0049-p000 -->


In [25]:
verified_run = RunTrace.load(TRACES_DIR / "verified_research_20261002T193656Z_f169a52a0918.json")
print(verified_run.question_obj().question)
print(f"Status: {verified_run.status}; p={verified_run.final_probability}")
print(f"Model calls: {len(verified_run.llm_calls)}; tokens: {verified_run.total_tokens():,}")
print(verified_run.agent_artifacts()[0].rationale)
print(show_agent_timeline(verified_run.agent_artifacts()[0]))

Will the Federal Reserve lower the upper bound of its federal funds target range at the October 27-28, 2026 FOMC meeting?
Status: accepted; p=0.07
Model calls: 4; tokens: 8,613
Based on evidence from https://www.cnbc.com/2026/09/16/fed-rate-decision-september-2026.html, the Fed hiked rates to 3.75%-4% at the September 2026 meeting, with 16 of 18 participants expecting another rate hike later in 2026. From https://streetstats.finance/rates/fedfunds, futures markets as of late September 2026 are pricing the rate to rise to ~4.2% by December 2026, indicating further tightening expected. The September 2026 FOMC projections (https://www.federalreserve.gov/monetarypolicy/fomcprojtabl20260916.htm) show a median federal funds rate of 4.1% for end-2026, consistent with one more hike. All signals point to continued tightening, making a rate cut (lower upper bound) at the October 27-28 meeting extremely unlikely. Probability of YES is approximately 7%.
  VERIFICATION_RESEARCH  ·  p_yes=0.07  stat